In [10]:
import socket
import json

In [13]:
import socket
import json

#------------------------------
# 1. 서버 네트워크 설정
#------------------------------
HOST = '192.168.0.46'
PORT = 58129

#------------------------------
# 2. 분석 함수 정의
#------------------------------
def analyze_text(request):
    mode = request.get("mode")
    text = request.get("text", "")

    if mode == "length":
        return {
            "result": len(text),
            "desc": f"문자 길이는 {len(text)}자 입니다."  # f-string 안에 [] 말고 {} 써야해요!
        }
    elif mode == "sentiment":
        if any(word in text for word in ["좋아", "행복", "멋져", "훌륭", "사랑"]):
            sentiment = "positive"
        elif any(word in text for word in ["싫어", "불만", "짜증", "나빠", "불행"]):
            sentiment = "negative"
        else:
            sentiment = "neutral"
        return {
            "result": sentiment,
            "desc": f"감정 분석 결과: {sentiment}"
        }
    elif mode == "keywords":
        keywords = ["AI", "고장", "불량", "스크래치", "찍힘"]
        found = [k for k in keywords if k in text]
        return {
            "result": found,
            "desc": f"키워드 발견: {', '.join(found) if found else '없음'}"
        }
    else:
        return {"error": f"지원하지 않는 분석 모드입니다. : {mode}"}

#------------------------------
# 3. TCP 서버 생성 및 설정
#------------------------------
server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.bind((HOST, PORT))
server_socket.listen()
print(f"AI 서버 실행 중..........({HOST}:{PORT})")

#------------------------------
# 4. 클라이언트 연결 수락
#------------------------------
client_socket, addr = server_socket.accept()
print(f"클라이언트 연결됨: {addr}")

#------------------------------
# 5. 메세지 송수신 응답 루프
#------------------------------
while True:
    data = client_socket.recv(2048).decode()
    if not data:
        print("클라이언트 연결 종료 감지")
        break
    if data.lower() == 'exit':
        print("클라이언트 종료 요청 수신")
        break
    try:
        request = json.loads(data)        # ← data를 json으로 파싱
        result = analyze_text(request)    # ← 파싱한 request를 함수에 넘김
    except json.JSONDecodeError:
        result = {"error": "잘못된 json 형식입니다."}

    response = json.dumps(result, ensure_ascii=False)  # jump → dumps
    client_socket.sendall(response.encode())

#------------------------------
# 6. 연결 종료
#------------------------------
client_socket.close()
server_socket.close()
print("AI 서버 종료 완료")

AI 서버 실행 중..........(192.168.0.46:58129)
클라이언트 연결됨: ('192.168.0.46', 62933)
클라이언트 연결 종료 감지
AI 서버 종료 완료
